# 01 - Environment, Hardware Feasibility, and Corpus

Legal AI / PEFT pivot. Everything needed to get from a bare checkout to a
machine that can serve three fine-tuned experts concurrently.

**Target hardware:** RTX 4070 Laptop GPU, 8 GB VRAM, Windows 11.

Run every cell from `legalai/`. All shell cells assume the venv interpreter
at `.venv\Scripts\python.exe` - **not** bare `python`. That distinction
cost several hours: on this machine bare `python` is the system interpreter,
which has `torch` but not `peft`/`accelerate`/`bitsandbytes`, so the server
started fine and then failed to load any model with a message saying to
install a package that was already installed (in the venv).

## 1.1 Interpreter and dependencies

In [ ]:
# The CUDA build of torch must come from PyTorch's own index. Installing
# plain `torch` from PyPI on Windows silently gives a CPU-only build and
# every model then runs on the CPU.
!.venv\Scripts\python.exe -m pip install torch --index-url https://download.pytorch.org/whl/cu126
!.venv\Scripts\python.exe -m pip install -r requirements-finetune.txt

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a')
print('bf16 supported', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else 'n/a')

# Verified working combination on the experiment machine:
#   torch 2.13.0+cu126, transformers 5.14.1, peft 0.20.0, trl 1.9.2,
#   bitsandbytes 0.50.0, accelerate 1.14.0, datasets 5.0.1

In [ ]:
# Confirm the interpreter you are about to use can see the whole stack.
# run_experiment.ps1 performs this same check and refuses to start without it.
import importlib.util as u
missing = [m for m in ('torch','transformers','peft','accelerate','bitsandbytes','datasets')
           if u.find_spec(m) is None]
print('missing:', missing or 'none')

## 1.2 Verify the models exist before designing around them

Two of the three models named in the original pivot plan could not be used,
and both were caught by querying the Hugging Face API rather than assuming:

| planned | outcome |
|---|---|
| `meta-llama/Llama-3.2-3B-Instruct` | exists but `gated: manual` - needs an accepted licence + `HF_TOKEN`. Used Unsloth's ungated mirror of the same weights. |
| `Qwen/Qwen2.5-3B-Instruct` | fine, ungated |
| **Ministral 3B** | **has no open weights at all.** Mistral open-weighted only the 8B of Les Ministraux. The repo `ministral/Ministral-3b-instruct` was created 2024-03-14, seven months *before* Mistral announced Ministral, so it is an unrelated model reusing the name. |

In [ ]:
import json, urllib.request

def hf_model(model_id):
    try:
        with urllib.request.urlopen('https://huggingface.co/api/models/' + model_id, timeout=25) as r:
            d = json.load(r)
        return dict(gated=d.get('gated'), downloads=d.get('downloads'),
                    created=d.get('createdAt'), author=d.get('author'))
    except Exception as exc:
        return f'FAIL {exc}'

for m in ['meta-llama/Llama-3.2-3B-Instruct',
          'unsloth/Llama-3.2-3B-Instruct',
          'Qwen/Qwen2.5-3B-Instruct',
          'mistralai/Ministral-3b-instruct',   # 401 - does not exist
          'ministral/Ministral-3b-instruct',   # third-party, predates the name
          'microsoft/Phi-3.5-mini-instruct',
          'ibm-granite/granite-3.1-2b-instruct']:
    print(f'{m:45s}', hf_model(m))

## 1.3 The hardware constraint that actually binds: grouped-query attention

The first replacement for the third slot was Phi-3.5-mini. All three models
**fitted in VRAM** with 252 MiB spare - so a weights-only check passes - but
concurrent inference was impossible, because their combined KV cache needed
roughly 2.1 GB.

Phi-3.5-mini has **no grouped-query attention** (32 KV heads), costing
384 KiB/token against Llama 3.2 3B's 112 and Qwen2.5 3B's 36.

**Attention architecture, not parameter count, decides how many experts fit.**
A residency check is not a sufficient feasibility test.

In [ ]:
# KV cache cost per token, computed from each model's own config.
import json, urllib.request

def kv_cost(model_id):
    url = f'https://huggingface.co/{model_id}/raw/main/config.json'
    with urllib.request.urlopen(url, timeout=25) as r:
        c = json.load(r)
    layers = c['num_hidden_layers']
    heads = c['num_attention_heads']
    kv_heads = c.get('num_key_value_heads', heads)
    head_dim = c.get('head_dim') or c['hidden_size'] // heads
    kib = layers * kv_heads * head_dim * 2 * 2 / 1024   # K and V, 2 bytes each
    return dict(layers=layers, kv_heads=kv_heads, head_dim=head_dim,
                gqa=kv_heads < heads, kib_per_token=round(kib))

for m in ['unsloth/Llama-3.2-3B-Instruct', 'Qwen/Qwen2.5-3B-Instruct',
          'microsoft/Phi-3.5-mini-instruct',      # 384 KiB/token - unusable here
          'ibm-granite/granite-3.1-2b-instruct']:  # 80 KiB/token - chosen
    print(f'{m:42s}', kv_cost(m))

### Final expert assignment

| role | base model | 4-bit VRAM | KV/token |
|---|---|---|---|
| legal | Llama 3.2 3B Instruct | 2206 MiB | 112 KiB |
| news | Qwen2.5 3B Instruct | 1992 MiB | 36 KiB |
| general_qa | Granite 3.1 2B Instruct | 1781 MiB | 80 KiB |
| | **all three resident** | **5979 MiB** | 912 MiB @ 4096 tok |

## 1.4 Step 0 - concurrent residency check (run this before anything else)

In [ ]:
# Loads all three models, generates from each, then runs three simultaneous
# generations from three threads (what the PARALLEL topology does), and
# budgets the KV cache. Writes finetune/vram_report.json.
!set LEGALAI_USE_ADAPTERS=0 && .venv\Scripts\python.exe finetune\check_vram.py --concurrent

**Measured result: PASS** - 5979 MiB of weights, 1089 MiB free, 912 MiB of KV
cache at the concurrent-phase context length.

**Concurrency speedup: 1.07x**, against a theoretical ~3x. Three models sharing
one GPU serialise at the compute level no matter how the graph dispatches them.
This is reported in the paper: a concurrent topology chosen for latency reasons
should be validated against the serving hardware first.

## 1.5 Corpus

Two problems, both silent:

1. **The store held 44 stale news chunks, not the Act.** `AutoNewsFetcher` runs
   with `clear_existing=True`, so any interactive query with news fetching on
   replaces the statutory corpus. Every legal query would then abstain.
2. **The Act's download URL is dead.** The Parliament PDF returns HTTP 202 with
   an empty body and `eur-lex.europa.eu` answers non-browser clients with an
   AWS WAF challenge (`x-amzn-waf-action: challenge`).

Fix: ingest CELEX:32024R1689 - Regulation (EU) 2024/1689 as published in the
Official Journal - via the EU Publications Office **Cellar** endpoint, which is
built for machine access and requires an explicit `Accept-Language`.

In [ ]:
# Why the old sources fail, and the one that works.
import requests
for label, url, headers in [
    ('parliament (dead)', 'https://www.europarl.europa.eu/doceo/document/TA-9-2024-0138_EN.pdf', {}),
    ('eur-lex (WAF)',     'https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32024R1689', {}),
    ('cellar (works)',    'http://publications.europa.eu/resource/celex/32024R1689',
                          {'Accept': 'application/pdf', 'Accept-Language': 'eng'}),
]:
    try:
        r = requests.get(url, headers=headers, timeout=60)
        print(f'{label:20s} {r.status_code} len={len(r.content):>9} pdf={r.content[:5] == b"%PDF-"}')
    except Exception as exc:
        print(f'{label:20s} FAIL {type(exc).__name__}')

In [ ]:
# Ingest. --replace clears first, which is required when the store holds
# stale news chunks. embed.py RAISES rather than embedding an empty document
# if every source fails - do not lower that length floor.
!.venv\Scripts\python.exe embed.py --replace

# Expect: 599,775 characters -> 644 chunks

In [ ]:
# Always verify before a run. A few hundred chunks is right; tens means the
# corpus was overwritten by a news fetch.
import utils
print('chunks:', utils.get_db_document_count())

## 1.6 Ollama is still required

Generation is fully local via `transformers`, but **embeddings still run on
Ollama**. If Ollama is down, `RetrievalAgent` logs `Dense vector search failed`
and silently falls back to BM25-only - it completes, so a whole benchmark can
run on half its retrieval stack with nothing in the output saying so.

In [ ]:
import config, requests
from langchain_ollama import OllamaEmbeddings

r = requests.get(config.OLLAMA_BASE_URL.rstrip('/') + '/api/tags', timeout=8)
names = [m['name'] for m in r.json()['models']]
print('embed model present:', any(config.OLLAMA_EMBEDDING_MODEL in n for n in names))

vec = OllamaEmbeddings(model=config.OLLAMA_EMBEDDING_MODEL,
                       base_url=config.OLLAMA_BASE_URL).embed_query('high-risk AI system')
print('dense embedding dim:', len(vec))   # 768 for nomic-embed-text